In [1]:
# ── CELL 1: Import Libraries ──────────────────────────────────────────────────
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
# ── CELL 2: Data Loading ──────────────────────────────────────────────────────
# Load the Pima Indians Diabetes dataset
# Predicts whether a patient has diabetes (1) or not (0)
data = fetch_openml(name="diabetes", version=1, as_frame=True, parser="auto")

X = data.data
y = (data.target == "tested_positive").astype(int)
y = pd.Series(y.values, name="target")


In [3]:
# ── CELL 3: Data Exploration ──────────────────────────────────────────────────
print("Our Features (X):")
display(X.head())

print("\nOur Target (y):")
display(y.head())


Our Features (X):


,preg,plas,pres,skin,insu,mass,pedi,age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33



Our Target (y):


0    1
1    0
2    1
3    0
4    1
Name: target, dtype: int64

In [4]:
# ── CELL 4: Feature / Target Separation ──────────────────────────────────────
print("Features (X) shape:", X.shape)
print("Target  (y) shape :", y.shape)


Features (X) shape: (768, 8)
Target  (y) shape : (768,)


In [5]:
# ── CELL 5: Train / Test Split ────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Number of rows for training:", X_train.shape[0])
print("Number of rows for testing :", X_test.shape[0])


Number of rows for training: 614
Number of rows for testing : 154


In [6]:
# ── CELL 6: Baseline Model ───────────────────────────────────────────────────
baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_preds)
print(f"Baseline Model Accuracy: {baseline_accuracy * 100:.2f}%")


Baseline Model Accuracy: 64.29%


In [7]:
# ── CELL 7: Real Model Training ──────────────────────────────────────────────
real_model = LogisticRegression(max_iter=1000)
real_model.fit(X_train, y_train)

real_preds = real_model.predict(X_test)
real_accuracy = accuracy_score(y_test, real_preds)
print(f"Machine Learning Model Accuracy: {real_accuracy * 100:.2f}%")


Machine Learning Model Accuracy: 74.68%


In [8]:
# ── CELL 8: Predictions ──────────────────────────────────────────────────────
comparison = pd.DataFrame({
    "Actual"    : y_test.values[:10],
    "Predicted" : real_preds[:10],
    "Correct?"  : ["✅" if a == p else "❌"
                   for a, p in zip(y_test.values[:10], real_preds[:10])]
})
display(comparison)


,Actual,Predicted,Correct?
0,0,0,✅
1,0,0,✅
2,0,0,✅
3,0,0,✅
4,0,0,✅
5,0,0,✅
6,0,0,✅
7,0,1,❌
8,0,1,❌
9,0,1,❌


In [9]:
# ── CELL 9: Accuracy Evaluation ──────────────────────────────────────────────
print(f"Baseline Accuracy : {baseline_accuracy * 100:.2f}%")
print(f"Model Accuracy    : {real_accuracy * 100:.2f}%")
print(f"Improvement       : +{(real_accuracy - baseline_accuracy) * 100:.2f}%\n")

print(classification_report(y_test, real_preds,
                             target_names=["No Diabetes", "Diabetes"]))

cm_df = pd.DataFrame(
    confusion_matrix(y_test, real_preds),
    index   =["Actual: No Diabetes", "Actual: Diabetes"],
    columns =["Predicted: No Diabetes", "Predicted: Diabetes"]
)
display(cm_df)




Baseline Accuracy : 64.29%
Model Accuracy    : 74.68%
Improvement       : +10.39%

              precision    recall  f1-score   support

 No Diabetes       0.81      0.79      0.80        99
    Diabetes       0.64      0.67      0.65        55

    accuracy                           0.75       154
   macro avg       0.73      0.73      0.73       154
weighted avg       0.75      0.75      0.75       154



,Predicted: No Diabetes,Predicted: Diabetes
Actual: No Diabetes,78,21
Actual: Diabetes,18,37


In [10]:
# ── CELL 10: Interpretation ───────────────────────────────────────────────────
print("""
RESULTS INTERPRETATION

Dataset: Pima Indians Diabetes Dataset
        768 patients with 8 medical features and 2 classes (Diabetes / No Diabetes).

Features : Glucose, BMI, Age, Blood Pressure etc.

Baseline : Always guesses "No Diabetes" (most common).
        Scored 64.29% — our minimum bar to beat.

Model: Logistic Regression scored 74.68%.
        A +10.39% improvement over baseline.

Errors: 18 missed Diabetes cases (False Negatives)
        21 false alarms (False Positives).

Conclusion:
        74.68% is a reasonable result for real-world medical data.
        The dataset is small, which naturally limits
        accuracy.
""")


RESULTS INTERPRETATION

Dataset: Pima Indians Diabetes Dataset
        768 patients with 8 medical features and 2 classes (Diabetes / No Diabetes).

Features : Glucose, BMI, Age, Blood Pressure etc.

Baseline : Always guesses "No Diabetes" (most common).
        Scored 64.29% — our minimum bar to beat.

Model: Logistic Regression scored 74.68%.
        A +10.39% improvement over baseline.

Errors: 18 missed Diabetes cases (False Negatives)
        21 false alarms (False Positives).

Conclusion:
        74.68% is a reasonable result for real-world medical data.
        The dataset is small, which naturally limits
        accuracy.

